# Transformer-as-Dynamical-System: Language Stability Probes

This notebook builds a **practical experiment harness** for your idea: treat autoregressive decoding as a discrete-time dynamical system and track trajectory properties at each generated token.

We implement four families of per-step metrics inspired by interpretability/control analogies:

1. **Entropy trajectory** of next-token logits (uncertainty / convergence).
2. **Syntactic balance depth** (Dyck-like bracket invariant proxy).
3. **Semantic drift** via cosine similarity of current residual stream to prompt anchor.
4. **Attention concentration / sink / effective rank** diagnostics.

The goal is not to prove full correctness, but to measure which properties are empirically promising as stability monitors.

## Research grounding (short)

These probes are motivated by findings and practices from:

- Entropy/confidence dynamics in autoregressive decoding and calibration analyses.
- Bracket/stack-like syntactic probes for sequence models (Dyck-style formal language diagnostics).
- Representation trajectory analysis using cosine geometry in hidden spaces.
- Attention sink/long-context degeneration observations and low-rank attention behavior.

In this notebook, we keep implementation lightweight and model-agnostic in `transformer_lens` so you can adapt to SAE feature probes later.

In [ ]:
# Core setup
import os
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from transformer_lens import HookedTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Model configuration (matches your preferred stack)
MODEL_NAME = os.environ.get("TL_MODEL_NAME", "gemma-2-2b")

model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
model.eval()
print(f"Loaded model: {MODEL_NAME}")
print(f"n_layers={model.cfg.n_layers}, n_heads={model.cfg.n_heads}, d_model={model.cfg.d_model}")

## Metric definitions

For each generation step `t`:

- `H_t`: entropy of `softmax(logits_t)`.
- `depth_t`: bracket depth from decoded text (`()[]{}`), with mismatch counter.
- `sim_t`: cosine similarity between last-token residual stream and prompt-anchor residual mean.
- `attn_entropy_t`: average entropy of attention distribution over source positions (last query position, averaged across heads/layers).
- `sink_mass_t`: average attention mass on token position 0 (BOS sink proxy).
- `rank_t`: effective rank (entropy rank) of the last-layer mean attention matrix.

In [ ]:
@dataclass
class StepMetrics:
    step: int
    token_id: int
    token_str: str
    logit_entropy: float
    bracket_depth: int
    bracket_mismatch: int
    semantic_similarity: float
    attn_entropy: float
    sink_mass: float
    effective_rank: float


def shannon_entropy(probs: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    probs = probs.clamp_min(eps)
    return -(probs * probs.log()).sum(dim=-1)


def bracket_state(text: str) -> Tuple[int, int]:
    pairs = {')': '(', ']': '[', '}': '{'}
    opening = set(['(', '[', '{'])
    stack = []
    mismatch = 0
    for ch in text:
        if ch in opening:
            stack.append(ch)
        elif ch in pairs:
            if stack and stack[-1] == pairs[ch]:
                stack.pop()
            else:
                mismatch += 1
    return len(stack), mismatch


def effective_rank_from_matrix(mat: torch.Tensor, eps: float = 1e-12) -> float:
    # Entropy-based effective rank: exp(H(normalized singular values))
    s = torch.linalg.svdvals(mat)
    if float(s.sum()) <= eps:
        return 0.0
    p = s / s.sum()
    h = -(p * (p.clamp_min(eps)).log()).sum()
    return float(torch.exp(h).item())

In [ ]:
def compute_prompt_anchor_resid(model: HookedTransformer, prompt: str, anchor_layer: int = -1) -> torch.Tensor:
    tokens = model.to_tokens(prompt, prepend_bos=True).to(model.cfg.device)
    _, cache = model.run_with_cache(tokens)
    if anchor_layer < 0:
        anchor_layer = model.cfg.n_layers - 1
    resid_key = f"blocks.{anchor_layer}.hook_resid_post"
    resid = cache[resid_key][0]  # [seq, d_model]
    return resid.mean(dim=0).detach()


def run_trajectory(
    model: HookedTransformer,
    prompt: str,
    max_new_tokens: int = 80,
    temperature: float = 0.7,
    top_p: float = 0.95,
    anchor_layer: int = -1,
) -> Tuple[str, pd.DataFrame]:
    model_device = model.cfg.device
    tokens = model.to_tokens(prompt, prepend_bos=True).to(model_device)
    anchor_vec = compute_prompt_anchor_resid(model, prompt, anchor_layer=anchor_layer)

    rows: List[StepMetrics] = []

    for t in range(max_new_tokens):
        logits, cache = model.run_with_cache(tokens)
        last_logits = logits[:, -1, :]

        if temperature <= 0:
            next_token = torch.argmax(last_logits, dim=-1)
        else:
            scaled = last_logits / max(temperature, 1e-6)
            probs = F.softmax(scaled, dim=-1)

            if top_p < 1.0:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
                csum = sorted_probs.cumsum(dim=-1)
                mask = csum > top_p
                mask[..., 0] = False
                sorted_probs = sorted_probs.masked_fill(mask, 0)
                sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
                idx = torch.multinomial(sorted_probs, num_samples=1)
                next_token = torch.gather(sorted_idx, -1, idx).squeeze(-1)
            else:
                next_token = torch.multinomial(probs, num_samples=1).squeeze(-1)

        # Metric 1: logit entropy
        token_probs = F.softmax(last_logits, dim=-1)
        logit_entropy = float(shannon_entropy(token_probs)[0].item())

        # Metric 2: bracket depth/mismatch
        next_token_id = int(next_token.item())
        next_token_str = model.to_string(torch.tensor([next_token_id], device=model_device))
        decoded_so_far = model.to_string(tokens[0]) + next_token_str
        depth, mismatch = bracket_state(decoded_so_far)

        # Metric 3: semantic drift similarity from final residual
        layer_for_resid = model.cfg.n_layers - 1 if anchor_layer < 0 else anchor_layer
        resid_key = f"blocks.{layer_for_resid}.hook_resid_post"
        last_resid = cache[resid_key][0, -1, :].detach()
        sim = F.cosine_similarity(last_resid, anchor_vec, dim=0)
        semantic_similarity = float(sim.item())

        # Metric 4a/4b: attention entropy + sink mass (position 0) using last query row
        attn_entropies = []
        sink_masses = []
        for layer in range(model.cfg.n_layers):
            patt = cache[f"blocks.{layer}.attn.hook_pattern"][0]  # [heads, q, k]
            last_q = patt[:, -1, :]  # [heads, k]
            ent = shannon_entropy(last_q)
            attn_entropies.append(ent.mean())
            sink_masses.append(last_q[:, 0].mean())
        attn_entropy = float(torch.stack(attn_entropies).mean().item())
        sink_mass = float(torch.stack(sink_masses).mean().item())

        # Metric 4c: effective rank on last-layer, head-averaged full attention matrix
        patt_last = cache[f"blocks.{model.cfg.n_layers - 1}.attn.hook_pattern"][0]  # [heads, q, k]
        avg_mat = patt_last.mean(dim=0)  # [q, k]
        effective_rank = effective_rank_from_matrix(avg_mat)

        rows.append(
            StepMetrics(
                step=t,
                token_id=next_token_id,
                token_str=next_token_str,
                logit_entropy=logit_entropy,
                bracket_depth=depth,
                bracket_mismatch=mismatch,
                semantic_similarity=semantic_similarity,
                attn_entropy=attn_entropy,
                sink_mass=sink_mass,
                effective_rank=effective_rank,
            )
        )

        tokens = torch.cat([tokens, next_token.unsqueeze(0)], dim=1)

        # optional stop on EOS if present
        if next_token_id == model.tokenizer.eos_token_id:
            break

    generated_text = model.to_string(tokens[0])
    df = pd.DataFrame([r.__dict__ for r in rows])
    return generated_text, df

In [ ]:
def plot_trajectory(df: pd.DataFrame, title: str):
    fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
    ax = axes.ravel()

    ax[0].plot(df.step, df.logit_entropy)
    ax[0].set_title("Logit entropy")

    ax[1].plot(df.step, df.semantic_similarity)
    ax[1].set_title("Semantic similarity to prompt anchor")

    ax[2].plot(df.step, df.bracket_depth)
    ax[2].set_title("Bracket depth")

    ax[3].plot(df.step, df.bracket_mismatch)
    ax[3].set_title("Bracket mismatch count")

    ax[4].plot(df.step, df.attn_entropy)
    ax[4].set_title("Attention entropy (avg layers/heads)")

    ax[5].plot(df.step, df.sink_mass)
    ax[5].plot(df.step, df.effective_rank)
    ax[5].legend(["Sink mass (pos=0)", "Effective rank (last-layer avg)"])
    ax[5].set_title("Attention sink/rank")

    for a in ax:
        a.grid(alpha=0.3)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Experiment A: low vs high temperature stability

Hypothesis:

- Lower temperature should usually produce lower/steadier entropy and less semantic drift.
- High temperature may increase entropy, sink effects, and topic drift.

In [ ]:
prompt = """Write a concise explanation of why gradient descent converges on a strongly convex quadratic function."""

gen_low, df_low = run_trajectory(
    model,
    prompt,
    max_new_tokens=80,
    temperature=0.3,
    top_p=0.9,
)

gen_high, df_high = run_trajectory(
    model,
    prompt,
    max_new_tokens=80,
    temperature=1.2,
    top_p=0.95,
)

print("LOW-T output preview:
", gen_low[:800], "
")
print("HIGH-T output preview:
", gen_high[:800])

In [ ]:
plot_trajectory(df_low, "Low temperature trajectory")
plot_trajectory(df_high, "High temperature trajectory")

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "mean_logit_entropy",
        "std_logit_entropy",
        "mean_semantic_similarity",
        "final_semantic_similarity",
        "max_bracket_depth",
        "final_bracket_depth",
        "final_bracket_mismatch",
        "mean_attn_entropy",
        "mean_sink_mass",
        "mean_effective_rank",
    ],
    "low_temp": [
        df_low.logit_entropy.mean(),
        df_low.logit_entropy.std(),
        df_low.semantic_similarity.mean(),
        df_low.semantic_similarity.iloc[-1],
        df_low.bracket_depth.max(),
        df_low.bracket_depth.iloc[-1],
        df_low.bracket_mismatch.iloc[-1],
        df_low.attn_entropy.mean(),
        df_low.sink_mass.mean(),
        df_low.effective_rank.mean(),
    ],
    "high_temp": [
        df_high.logit_entropy.mean(),
        df_high.logit_entropy.std(),
        df_high.semantic_similarity.mean(),
        df_high.semantic_similarity.iloc[-1],
        df_high.bracket_depth.max(),
        df_high.bracket_depth.iloc[-1],
        df_high.bracket_mismatch.iloc[-1],
        df_high.attn_entropy.mean(),
        df_high.sink_mass.mean(),
        df_high.effective_rank.mean(),
    ]
})
summary

## Experiment B: constrained syntax task (code completion)

We test whether Dyck-like invariants are useful in a domain with clearer structure (code).

Interpretation guideline:

- Promising monitor: depth returns to near zero and mismatch remains low by end of generation.
- Warning sign: steadily increasing depth or frequent mismatch growth.

In [ ]:
code_prompt = """Complete this Python function and keep brackets balanced.

def running_average(xs):
    total = 0.0
    out = []
    for i, x in enumerate(xs):
"""

gen_code, df_code = run_trajectory(
    model,
    code_prompt,
    max_new_tokens=120,
    temperature=0.4,
    top_p=0.9,
)

print(gen_code[:1200])
plot_trajectory(df_code, "Code prompt trajectory")

pd.DataFrame({
    "final_depth": [int(df_code.bracket_depth.iloc[-1])],
    "final_mismatch": [int(df_code.bracket_mismatch.iloc[-1])],
    "max_depth": [int(df_code.bracket_depth.max())],
    "entropy_trend": [float(np.polyfit(df_code.step, df_code.logit_entropy, deg=1)[0])],
    "semantic_similarity_trend": [float(np.polyfit(df_code.step, df_code.semantic_similarity, deg=1)[0])],
})

## Optional: connect with SAE features later

You can extend this by extracting SAE activations per step and testing whether specific SAE feature trajectories correlate with:

- impending bracket mismatch,
- sudden semantic drift,
- sustained high entropy regimes,
- sink-mass spikes.

That gives a path from coarse monitors to mechanistic hypotheses.

## Practical next steps

1. Build a benchmark suite of prompts with known structural invariants (JSON, code, math proofs with delimiters).
2. Fit threshold-based alarms (`entropy`, `semantic_similarity`, `sink_mass`) and evaluate precision/recall for failure prediction.
3. Compare across model scales and quantization settings.
4. Add MMLU-style multiple-choice trajectories (your existing setup) as a second task family.